<a href="https://colab.research.google.com/github/ZALER-01/AngularPractice/blob/main/stock_trading_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit yfinance pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 56.9 MB/s eta 0:00:00


In [4]:
%%writefile app.py
import os
os.environ["YFINANCE_USE_CURL"] = "False"

import streamlit as st
import yfinance as yf
import pandas as pd

def calculate_rsi(data, window=14):
    delta = data['Close'].diff()
    gain = delta.clip(lower=0).rolling(window).mean()
    loss = -delta.clip(upper=0).rolling(window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def get_support_level(data):
    return float(data['Low'].tail(20).min())

def calculate_target_stoploss(price, support):
    stop_loss = support * 0.98
    risk = price - support
    target = price + (2 * risk)
    return round(target, 2), round(stop_loss, 2)

def generate_signal(curr):
    sma50 = curr['SMA50'].item()
    sma200 = curr['SMA200'].item()
    rsi = curr['RSI'].item()

    if sma50 > sma200 and rsi < 70:
        return "🟢 BUY"
    elif sma50 < sma200 and rsi > 40:
        return "🔴 SELL"
    else:
        return "🟡 HOLD"

def fetch_data(ticker):
    try:
        data = yf.download(ticker, period="1y", interval="1d", progress=False, auto_adjust=True)
        if data is None or data.empty:
            return None
        return data
    except:
        return None

def calculate_market_return(nifty, buy_date):
    try:
        start_price = nifty[nifty.index >= buy_date].iloc[0]['Close'].item()
        current_price = nifty.iloc[-1]['Close'].item()
        return ((current_price - start_price) / start_price) * 100
    except:
        return None

st.title("📈 Stock Trading Assistant")

stock = st.text_input("Stock", "HDFCBANK.NS")
buy_price = st.number_input("Buying Price", value=750.0)
buy_date = st.date_input("Buying Date")

if st.button("Analyze"):

    data = fetch_data(stock)

    if data is None:
        st.error("Invalid stock")
    else:
        nifty = fetch_data("^NSEI")

        data['SMA50'] = data['Close'].rolling(50).mean()
        data['SMA200'] = data['Close'].rolling(200).mean()
        data['RSI'] = calculate_rsi(data)

        data = data.dropna()
        latest = data.iloc[-1]

        current_price = latest['Close'].item()
        stock_return = ((current_price - buy_price) / buy_price) * 100

        market_return = calculate_market_return(nifty, str(buy_date)) if nifty is not None else None

        support = get_support_level(data)
        target, stop_loss = calculate_target_stoploss(current_price, support)
        signal = generate_signal(latest)

        st.subheader("Result")
        st.write(f"Price: {round(current_price,2)}")
        st.write(f"Return: {round(stock_return,2)}%")
        st.write(f"Market Return: {round(market_return,2) if market_return else 'N/A'}")
        st.write(f"RSI: {round(latest['RSI'].item(),2)}")
        st.write(f"Support: {support}")
        st.write(f"Target: {target}")
        st.write(f"Stop Loss: {stop_loss}")
        st.write(f"Signal: {signal}")

Overwriting app.py


In [13]:
from pyngrok import ngrok

ngrok.set_auth_token("2A6KpZm2XvCJTAL6HgAcp_4otSd3AeaCwfyTarUozEv")

In [14]:
from pyngrok import ngrok
!streamlit run app.py &>/dev/null &
public_url = ngrok.connect(80)
public_url

PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: Your account may not run more than 3 endpoints over a single ngrok agent session.\nThe endpoints already running on this session are:\ntn_3BQY1iNkEz1klr8f4j4jDDE3bk6, tn_3BQYCUL4il2evGC7pAJkgClpPnh, tn_3BQYFdW9LoAeQBaXpsQKCD53zO8.\nUpgrade to a Pay-as-you-go plan at: https://dashboard.ngrok.com/billing/choose-a-plan?plan=paygo\r\n\r\nERR_NGROK_324\r\n"}}
